In [1]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [4]:
# create the training set of bigrams (x,y)
xs, ys = [], []

for w in words[:1]:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    print(ch1, ch2)
    xs.append(ix1)
    ys.append(ix2)
    
xs = torch.tensor(xs)
ys = torch.tensor(ys)

. e
e m
m m
m a
a .


In [5]:
# randomly initialize 27 neurons' weights. each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W1 = torch.randn((27, 27), generator=g)

In [6]:
xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
logits = xenc @ W1 # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
# btw: the last 2 lines here are together called a 'softmax'

In [7]:
probs.shape

torch.Size([5, 27])

In [8]:
xs

tensor([ 0,  5, 13, 13,  1])

In [9]:
ys

tensor([ 5, 13, 13,  1,  0])

In [10]:
import time

In [ ]:
# create the dataset
xs, ys = [], []
for w in words[:1]:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    xs.append(ix1)
    ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W1 = torch.randn((27, 27), generator=g, requires_grad=True)

W1_clone = W1.clone()
W1_grad_clone = None

arr = []
arr_losses = []
arr_main_loss = []
arr_synthetic_loss = []
arr_test_loss = []

z = torch.ones(W1[xs].view(-1).shape[0], 2)

# second layer
W2 = torch.randn((27, 500), generator=g, requires_grad=True)
W3 = torch.randn((500, 100), generator=g, requires_grad=True)
W4 = torch.randn((100, 27), generator=g, requires_grad=True)
act = torch.nn.ReLU()

build_2d_plot_flag = 0

# optimizer - settings from makemore
#params = {W1, W2, W3, W4}
params = {W1}
print("number of layers: " , len(params))
optimizer = torch.optim.SGD(params, lr=0.1, momentum=0.9)
#optimizer = torch.optim.AdamW(params, lr=5e-4, weight_decay=0.01, betas=(0.9, 0.99), eps=1e-8)
#optimizer = torch.optim.AdamW(params, lr=0.1, weight_decay=0.01, betas=(0.9, 0.99), eps=1e-8)

print()
print( "--- parameters in layers --- ")
total_sum = 0
for p in params:
    print(p.view(-1).shape)
    total_sum += p.view(-1).shape[0]
print(20*'-')
print( "TOTAL: ", total_sum)
print(20*'-')

number of examples:  5
number of layers:  1


In [ ]:
xs, ys

In [ ]:
words[:3]

In [ ]:
# gradient descent
start_timer = time.perf_counter()
for k in range(50):
    
    # forward pass
    xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
    #print(xenc)
    logits = xenc @ W1 # predict log-counts
    #logits = act(logits)
    #logits = logits @ W1
    #l1 = torch.tanh(logits)
    #l2 = torch.tanh(logits @ W2) @ W3
    #print(l2)
    #break
    #l3 = l2 @ W3
    #logits = torch.tanh(l2) @ W4
    #print(logits.shape)
    #print(logits)
    #break

    """
    # num samples = 1, 2, 3 - most of tests for this is for for w in words[:1]: -> emma\n
    if num < 20:
        # compute expensive - MEMORY!!!
        index = 0
        for i in W1[xs].view(-1)[:]:
            #print(i)
            #print(z[index] * i )
            z[index] *= i
            index += 1
        arr.append(z.clone())
        # check flag
        if build_2d_plot_flag == 0:
            print(" 2d plot builder active")
            build_2d_plot_flag = 1
    """
    
    counts = logits.exp() # counts, equivalent to N
    probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
    #print(probs)
    #print(-probs[torch.arange(num), ys].log())

    arr_losses.append(-probs[torch.arange(num), ys].log().detach().numpy())
    
    #loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W1**2).mean()

    loss = -probs[torch.arange(num), ys].log().mean() # I NEED RAW .mean() only

    #arr_main_loss.append(loss.item()) # collect before BACKWARD(). call
    #arr_synthetic_loss.append(torch.tensor(arr_losses).mean().item())

    #test_loss = F.cross_entropy(logits, ys)
    #arr_test_loss.append(test_loss.item())
    
    #print(-probs[torch.arange(num), ys].log())
    
    print(k , loss.item())
    
    # backward pass
    #W1.grad = None # set to zero the gradient
    #W2.grad = None 

    optimizer.zero_grad()
    
    loss.backward()

    if (k == 0):
        W1_grad_clone = W1.grad.clone()

    #break
    
    # LR form 50 to 1
    LR = 1
    # update
    #W1.data += -(LR) * W1.grad
    #W2.data += -(LR) * W2.grad
    
    optimizer.step()
    
end_timer = time.perf_counter()

In [ ]:
loss

In [ ]:
print(f"gap in {end_timer - start_timer:0.4f} seconds")

In [ ]:
step = 0
for p in range(len(arr_losses)):
    plt.plot(torch.tensor(arr_losses)[:, step])
    if step == num-1:
        step = 0
    step += 1
plt.grid()

In [ ]:
len(arr_losses)

In [ ]:
torch.tensor(arr_losses)[:, 3].shape

In [ ]:
torch.tensor(arr_losses)[:, 3]

In [ ]:
loss

In [ ]:
torch.tensor(arr_losses)[-1]

In [ ]:
torch.tensor(arr_losses)[-1].mean()